<a href="https://colab.research.google.com/github/VISHWAJA-028/ML_Multimodality/blob/main/pcos_predictor_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import importlib
import io
import os
import pickle
import platform
import re
import tempfile
import warnings

import numpy as np
import pandas as pd


BUNDLE_FORMAT = 1

DISCLAIMER = (
    "Screening aid only - not a medical diagnosis. "
    "Please confirm with a qualified clinician."
)


# ============================================================
# KEY FIELDS
# ============================================================

KEY_FIELDS = [
    "hair_growth_Y_N",
    "Weight_gain_Y_N",
    "Skin_darkening_Y_N",
    "Follicle_No_R",
    "Follicle_No_L",
    "AMH_ng_mL"
]

_DERIVED = {
    "Follicle_Total",
    "Avg_F_Size_Mean"
}

_DUMMY_PREFIXES = (
    "Blood_Group_",
    "Cycle_R_I_",
    "BMI_Category_"
)

_IGNORED_INPUTS = {
    "Sl_No",
    "Patient_File_No",
    "Source_Dataset",
    "PCOS_Y_N"
}


# ============================================================
# BLOOD GROUP / CYCLE MAPPINGS
# ============================================================

_BG_COL = {
    "A+": "Blood_Group_A",
    "A-": "Blood_Group_A_1",
    "AB+": "Blood_Group_AB",
    "AB-": "Blood_Group_AB_1",
    "B+": "Blood_Group_B",
    "B-": "Blood_Group_B_1",
    "O+": "Blood_Group_O",
    "O-": "Blood_Group_O_1"
}

_BG_CODES = {
    "11": "A+",
    "12": "A-",
    "13": "B+",
    "14": "B-",
    "15": "O+",
    "16": "O-",
    "17": "AB+",
    "18": "AB-"
}

_CYCLE_CODES = {
    "2": "R",
    "4": "I",
    "R": "R",
    "I": "I",
    "REGULAR": "R",
    "IRREGULAR": "I"
}


# ============================================================
# 1. FEATURE ENGINEERING
# ============================================================

def clean_name(col):
    """
    Convert original column names into cleaned names.

    Example:
    Follicle No. (L) -> Follicle_No_L
    """

    c = re.sub(
        r"[^A-Za-z0-9_]",
        "_",
        str(col)
    )

    c = re.sub(
        r"_{2,}",
        "_",
        c
    )

    return c.strip("_")


def _as_code(v):
    """
    Convert values such as:
    12
    12.0
    '12'

    into:
    '12'
    """

    if v is None:
        return None

    if isinstance(v, float) and np.isnan(v):
        return None

    s = str(v).strip()

    try:
        f = float(s)

        if f == int(f):
            return str(int(f))

    except ValueError:
        pass

    return s.upper()


def _yn(v):

    if v is None:
        return np.nan

    if v is pd.NA:
        return np.nan

    if isinstance(v, float) and np.isnan(v):
        return np.nan

    return {
        "Y": 1,
        "YES": 1,
        "1": 1,
        "1.0": 1,
        "TRUE": 1,

        "N": 0,
        "NO": 0,
        "0": 0,
        "0.0": 0,
        "FALSE": 0
    }.get(
        str(v).strip().upper(),
        np.nan
    )


def _norm_blood_group(v):

    s = _as_code(v)

    if s is None:
        return None

    s = _BG_CODES.get(
        s,
        s
    )

    if s in _BG_COL:
        return s

    return None


def _norm_cycle(v):

    s = _as_code(v)

    if s is None:
        return None

    return _CYCLE_CODES.get(s)


def _fix_double_decimal(x):

    s = str(x)

    if s.count(".") > 1:

        i = s.index(".")

        s = (
            s[:i + 1]
            + s[i + 1:].replace(".", "")
        )

    return s


def _bmi_category(b):

    if pd.isna(b):
        return None

    if b < 18.5:
        return "Underweight"

    if b < 25:
        return "Normal"

    if b < 30:
        return "Overweight"

    return "Obese"


def engineer_features(raw, columns):
    """
    Convert user input into exactly the same
    feature structure used during model training.
    """

    if isinstance(
        raw,
        pd.DataFrame
    ):

        df = raw.copy()

    elif isinstance(
        raw,
        pd.Series
    ):

        df = raw.to_frame().T

    elif isinstance(
        raw,
        dict
    ):

        df = pd.DataFrame([raw])

    else:

        raise TypeError(
            "features must be a dict, "
            "pandas Series or DataFrame"
        )

    df = df.reset_index(
        drop=True
    )

    df.columns = [
        clean_name(c)
        for c in df.columns
    ]

    df = df.loc[
        :,
        ~df.columns.duplicated()
    ]

    blank = pd.Series(
        np.nan,
        index=df.index,
        dtype=object
    )

    dummies = [
        c
        for c in columns
        if c.startswith(
            _DUMMY_PREFIXES
        )
    ]

    yn_cols = [
        c
        for c in columns
        if c.endswith("_Y_N")
    ]

    numeric = [
        c
        for c in columns
        if c not in dummies
        and c not in yn_cols
        and c not in _DERIVED
    ]

    out = pd.DataFrame(
        index=df.index
    )

    # --------------------------------------------------------
    # Numeric columns
    # --------------------------------------------------------

    for c in numeric:

        s = (
            df[c]
            if c in df.columns
            else blank
        )

        if c == "II_beta_HCG_mIU_mL":

            s = s.map(
                _fix_double_decimal
            )

        out[c] = pd.to_numeric(
            s,
            errors="coerce"
        ).astype(float)

    # --------------------------------------------------------
    # Derived features
    # --------------------------------------------------------

    if {
        "BMI",
        "Weight_Kg",
        "Height_Cm"
    } <= set(out.columns):

        out["BMI"] = out["BMI"].fillna(
            out["Weight_Kg"]
            / (out["Height_Cm"] / 100) ** 2
        )

    if {
        "Waist_Hip_Ratio",
        "Waist_inch",
        "Hip_inch"
    } <= set(out.columns):

        out["Waist_Hip_Ratio"] = (
            out["Waist_Hip_Ratio"].fillna(
                out["Waist_inch"]
                / out["Hip_inch"].replace(
                    0,
                    np.nan
                )
            )
        )

    if {
        "FSH_LH",
        "FSH_mIU_mL",
        "LH_mIU_mL"
    } <= set(out.columns):

        out["FSH_LH"] = (
            out["FSH_LH"].fillna(
                out["FSH_mIU_mL"]
                / out["LH_mIU_mL"].replace(
                    0,
                    np.nan
                )
            )
        )

    # --------------------------------------------------------
    # Follicle features
    # --------------------------------------------------------

    if (
        "Follicle_Total" in columns
        and {
            "Follicle_No_L",
            "Follicle_No_R"
        } <= set(out.columns)
    ):

        out["Follicle_Total"] = (
            out["Follicle_No_L"]
            + out["Follicle_No_R"]
        )

    if (
        "Avg_F_Size_Mean" in columns
        and {
            "Avg_F_size_L_mm",
            "Avg_F_size_R_mm"
        } <= set(out.columns)
    ):

        out["Avg_F_Size_Mean"] = (
            out[
                [
                    "Avg_F_size_L_mm",
                    "Avg_F_size_R_mm"
                ]
            ].mean(axis=1)
        )

    # --------------------------------------------------------
    # Yes / No columns
    # --------------------------------------------------------

    for c in yn_cols:

        out[c] = (
            df[c]
            if c in df.columns
            else blank
        ).map(
            _yn
        ).astype(float)

    # --------------------------------------------------------
    # Blood group / cycle / BMI
    # --------------------------------------------------------

    bg = (
        df["Blood_Group"]
        if "Blood_Group" in df.columns
        else blank
    ).map(
        _norm_blood_group
    )

    cy = (
        df["Cycle_R_I"]
        if "Cycle_R_I" in df.columns
        else blank
    ).map(
        _norm_cycle
    )

    bc = (
        out["BMI"].map(
            _bmi_category
        )
        if "BMI" in out.columns
        else blank
    )

    # --------------------------------------------------------
    # Blood group dummy variables
    # --------------------------------------------------------

    for label, col in _BG_COL.items():

        out[col] = (
            bg == label
        ).astype(float)

    out["Blood_Group_nan"] = (
        bg.isna()
    ).astype(float)

    # --------------------------------------------------------
    # Cycle dummy variables
    # --------------------------------------------------------

    out["Cycle_R_I_I"] = (
        cy == "I"
    ).astype(float)

    out["Cycle_R_I_R"] = (
        cy == "R"
    ).astype(float)

    out["Cycle_R_I_nan"] = (
        cy.isna()
    ).astype(float)

    # --------------------------------------------------------
    # BMI dummy variables
    # --------------------------------------------------------

    for cat in (
        "Normal",
        "Obese",
        "Overweight",
        "Underweight"
    ):

        out[
            f"BMI_Category_{cat}"
        ] = (
            bc == cat
        ).astype(float)

    out["BMI_Category_nan"] = (
        bc.isna()
    ).astype(float)

    # --------------------------------------------------------
    # Match training columns exactly
    # --------------------------------------------------------

    out = out.reindex(
        columns=list(columns)
    )

    fields = (
        numeric
        + yn_cols
        + [
            "Blood_Group",
            "Cycle_R_I"
        ]
    )

    info = {
        "fields": fields,
        "missing": None,
        "unknown": None
    }

    # --------------------------------------------------------
    # Missing / unknown fields
    # --------------------------------------------------------

    if len(df) == 1:

        have = {
            c: bool(
                pd.notna(
                    out.at[0, c]
                )
            )
            for c in numeric + yn_cols
        }

        have["Blood_Group"] = bool(
            pd.notna(
                bg.iloc[0]
            )
        )

        have["Cycle_R_I"] = bool(
            pd.notna(
                cy.iloc[0]
            )
        )

        info["missing"] = [
            c
            for c in fields
            if not have[c]
        ]

        known = (
            set(columns)
            | {
                "Blood_Group",
                "Cycle_R_I"
            }
            | _IGNORED_INPUTS
        )

        info["unknown"] = [
            c
            for c in df.columns
            if c not in known
        ]

    return out, info


# ============================================================
# 2. THRESHOLD HELPER
# ============================================================

def tune_threshold(
    y_true_pcos,
    p_pcos,
    min_recall=0.95
):

    from sklearn.metrics import roc_curve

    fpr, tpr, thr = roc_curve(
        y_true_pcos,
        p_pcos
    )

    ok = np.where(
        tpr >= min_recall
    )[0]

    if len(ok) == 0:

        raise ValueError(
            "min_recall not reachable"
        )

    return float(
        thr[
            ok[
                np.argmin(
                    fpr[ok]
                )
            ]
        ]
    )


# ============================================================
# 3. VERSION INFORMATION
# ============================================================

def _versions():

    v = {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__
    }

    for m in (
        "sklearn",
        "lightgbm",
        "xgboost",
        "tensorflow"
    ):

        try:

            v[m] = (
                importlib
                .import_module(m)
                .__version__
            )

        except Exception:

            pass

    return v


# ============================================================
# 4. CREATE BUNDLE
# ============================================================

def create_bundle(
    path,
    tab_model,
    tab_imputer,
    tab_scaler,
    tab_columns,
    keras_model_path,
    img_size=(224, 224),
    image_p_is_pcos=False,
    image_threshold=0.5,
    image_tta=1,
    tab_threshold=0.5,
    meta=None
):

    with open(
        keras_model_path,
        "rb"
    ) as f:

        keras_bytes = f.read()

    bundle = {

        "format": BUNDLE_FORMAT,

        "tabular": {

            "model": tab_model,

            "imputer": tab_imputer,

            "scaler": tab_scaler,

            "columns": list(
                tab_columns
            ),

            "threshold": float(
                tab_threshold
            )
        },

        "image": {

            "keras_bytes": keras_bytes,

            "img_size": tuple(
                img_size
            ),

            # Current image model:
            # infected = 0
            # noninfected = 1
            #
            # Therefore CNN output is
            # P(noninfected)

            "p_is_pcos": bool(
                image_p_is_pcos
            ),

            "threshold": float(
                image_threshold
            ),

            "tta": int(
                image_tta
            )
        },

        "meta": {
            "versions": _versions(),
            **(meta or {})
        }
    }

    with open(
        path,
        "wb"
    ) as f:

        pickle.dump(
            bundle,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    return path


# ============================================================
# 5. PCOS PREDICTOR
# ============================================================

class PCOSPredictor:

    def __init__(
        self,
        bundle
    ):

        if bundle.get(
            "format"
        ) != BUNDLE_FORMAT:

            raise ValueError(
                "Unsupported bundle format"
            )

        self.tab = bundle[
            "tabular"
        ]

        self.img = bundle[
            "image"
        ]

        self.meta = bundle.get(
            "meta",
            {}
        )

        self._img_model = None


    # ========================================================
    # LOAD BUNDLE
    # ========================================================

    @classmethod
    def load(
        cls,
        path
    ):

        with open(
            path,
            "rb"
        ) as f:

            bundle = pickle.load(f)

        saved = (
            bundle
            .get("meta", {})
            .get("versions", {})
        )

        now = _versions()

        for lib in (
            "sklearn",
            "lightgbm",
            "tensorflow"
        ):

            if (
                lib in saved
                and lib in now
                and saved[lib] != now[lib]
            ):

                warnings.warn(
                    f"{lib} version differs "
                    f"(saved {saved[lib]}, "
                    f"now {now[lib]}); "
                    "results may change or "
                    "loading may fail."
                )

        return cls(
            bundle
        )


    # ========================================================
    # STRUCTURED MODEL
    # ========================================================

    def _tab_probs(
        self,
        X
    ):

        cols = self.tab[
            "columns"
        ]

        Xi = pd.DataFrame(
            self.tab["imputer"].transform(X),
            columns=cols,
            index=X.index
        )

        Xs = pd.DataFrame(
            self.tab["scaler"].transform(Xi),
            columns=cols,
            index=X.index
        )

        return self.tab[
            "model"
        ].predict_proba(
            Xs
        )[:, 1]


    def predict_structured(
        self,
        features
    ):

        X, info = engineer_features(
            features,
            self.tab["columns"]
        )

        if len(X) != 1:

            raise ValueError(
                "Use predict_structured_batch() "
                "for several rows"
            )

        p = float(
            self._tab_probs(X)[0]
        )

        thr = self.tab[
            "threshold"
        ]

        missing = info[
            "missing"
        ]

        completeness = (
            1
            - len(missing)
            / len(info["fields"])
        )

        key_missing = [
            c
            for c in KEY_FIELDS
            if c in missing
        ]

        warns = []

        if completeness < 0.6:

            warns.append(
                "Many fields are missing; "
                "they were filled with training "
                "medians, so treat this as "
                "low-confidence."
            )

        if key_missing:

            warns.append(
                "Strongest predictors not provided: "
                + ", ".join(key_missing)
            )

        if info["unknown"]:

            warns.append(
                "Ignored unrecognised fields "
                "(typo?): "
                + ", ".join(
                    info["unknown"]
                )
            )

        return {

            "label":
                "PCOS"
                if p >= thr
                else "Non-PCOS",

            "prob_pcos":
                round(p, 4),

            "threshold":
                thr,

            "input_completeness":
                round(
                    completeness,
                    2
                ),

            "missing_fields":
                missing,

            "warnings":
                warns
        }


    def predict_structured_batch(
        self,
        df
    ):

        X, _ = engineer_features(
            df,
            self.tab["columns"]
        )

        p = self._tab_probs(
            X
        )

        return pd.DataFrame({

            "prob_pcos":
                p.round(4),

            "label":
                np.where(
                    p >= self.tab["threshold"],
                    "PCOS",
                    "Non-PCOS"
                )
        })


    # ========================================================
    # IMAGE MODEL
    # ========================================================

    def _get_image_model(
        self
    ):

        if self._img_model is None:

            from tensorflow import keras

            with tempfile.NamedTemporaryFile(
                suffix=".keras",
                delete=False
            ) as f:

                f.write(
                    self.img[
                        "keras_bytes"
                    ]
                )

                tmp = f.name

            try:

                self._img_model = (
                    keras.models.load_model(
                        tmp,
                        compile=False
                    )
                )

            finally:

                os.remove(
                    tmp
                )

        return self._img_model


    def _load_image(
        self,
        image
    ):

        from PIL import Image

        if isinstance(
            image,
            (str, os.PathLike)
        ):

            im = Image.open(
                image
            )

        elif isinstance(
            image,
            (bytes, bytearray)
        ):

            im = Image.open(
                io.BytesIO(
                    image
                )
            )

        elif isinstance(
            image,
            Image.Image
        ):

            im = image

        elif isinstance(
            image,
            np.ndarray
        ):

            im = Image.fromarray(
                image.astype(
                    "uint8"
                )
            )

        else:

            raise TypeError(
                "image must be a path, "
                "bytes, PIL image or numpy array"
            )

        arr = np.asarray(
            im.convert("RGB")
        )

        size = tuple(
            self.img[
                "img_size"
            ]
        )

        try:

            import tensorflow as tf

            arr = (
                tf.image
                .resize(
                    arr,
                    size
                )
                .numpy()
            )

        except ImportError:

            arr = np.asarray(
                Image.fromarray(
                    arr
                ).resize(
                    size[::-1],
                    Image.BILINEAR
                )
            )

        return arr.astype(
            "float32"
        )


    def predict_image(
        self,
        image
    ):

        arr = self._load_image(
            image
        )

        n = max(
            1,
            int(
                self.img[
                    "tta"
                ]
            )
        )

        raw = float(
            self._get_image_model()
            .predict(
                np.repeat(
                    arr[None],
                    n,
                    axis=0
                ),
                verbose=0
            )
            .ravel()
            .mean()
        )

        # Current model output:
        # P(noninfected)
        #
        # Convert to P(PCOS)

        if self.img[
            "p_is_pcos"
        ]:

            p = raw

        else:

            p = 1.0 - raw

        thr = self.img[
            "threshold"
        ]

        is_pcos = (
            p >= thr
        )

        return {

            "label":
                "PCOS"
                if is_pcos
                else "Non-PCOS",

            "prob_pcos":
                round(
                    p,
                    4
                ),

            "threshold":
                thr,

            "passes_averaged":
                n
        }


    # ========================================================
    # COMBINED PREDICTION
    # ========================================================

    def predict(
        self,
        features=None,
        image=None,
        tab_weight=0.5
    ):

        if (
            features is None
            and image is None
        ):

            raise ValueError(
                "Give features=..., "
                "image=..., or both"
            )

        out = {
            "disclaimer":
                DISCLAIMER
        }

        t = (
            self.predict_structured(
                features
            )
            if features is not None
            else None
        )

        i = (
            self.predict_image(
                image
            )
            if image is not None
            else None
        )

        if t:
            out["structured"] = t

        if i:
            out["image"] = i

        if t and i:

            out["mode"] = "both"

        elif t:

            out["mode"] = "structured"

        else:

            out["mode"] = "image"

        # ----------------------------------------------------
        # Combine both models
        # ----------------------------------------------------

        if t and i:

            p = (
                tab_weight
                * t["prob_pcos"]
                +
                (1 - tab_weight)
                * i["prob_pcos"]
            )

            out["combined"] = {

                "label":
                    "PCOS"
                    if p >= 0.5
                    else "Non-PCOS",

                "prob_pcos":
                    round(
                        p,
                        4
                    ),

                "agree":
                    t["label"]
                    == i["label"],

                "note":
                    (
                        "Heuristic average: the two "
                        "models were trained on different "
                        "patients, so the weights are not "
                        "learned. When they disagree, "
                        "refer for clinical review."
                    )
            }

        return out


    # ========================================================
    # AUTO ROUTING
    # ========================================================

    def predict_any(
        self,
        x
    ):

        """
        Automatically route the input:

        dict / Series / DataFrame
            -> structured model

        anything else
            -> image model
        """

        if isinstance(
            x,
            (
                dict,
                pd.Series,
                pd.DataFrame
            )
        ):

            return self.predict(
                features=x
            )

        return self.predict(
            image=x
        )


    # ========================================================
    # DESCRIBE
    # ========================================================

    def describe(
        self
    ):

        print(
            "Structured columns:",
            len(
                self.tab[
                    "columns"
                ]
            ),
            "| threshold",
            self.tab[
                "threshold"
            ]
        )

        print(
            "Image size:",
            self.img[
                "img_size"
            ],

            "| P is PCOS:",
            self.img[
                "p_is_pcos"
            ],

            "| threshold:",
            self.img[
                "threshold"
            ],

            "| TTA:",
            self.img[
                "tta"
            ]
        )

        print(
            "Meta:",
            {
                k: v
                for k, v in self.meta.items()
                if k != "versions"
            }
        )

        print(
            "Saved with:",
            self.meta.get(
                "versions"
            )
        )